# E3′ — BERTimbau valida o FALCO, agora multi-semente (Colab / Databricks)

**O que este notebook faz:** treina o BERTimbau (classificador forte) em 5 braços
+ varredura de orçamento e mede na população reservada se o pipeline barato
(laço leve + oráculo LLM) entrega um modelo forte competitivo. Cada braço é UM
ajuste fino — o laço já rodou; os conjuntos estão prontos no repositório.

| Braço | Treino | Responde |
|---|---|---|
| A | 8.937 itens anotados pelo pipeline real (oráculo NIM) | o que o pipeline entrega |
| B | mesmos itens, rótulos gold | custo do ruído do oráculo (A−B) |
| C | 8.937 aleatórios, gold | valor da seleção (B−C) |
| D | pool 50k, gold | a régua (teto do pool) |
| E | 15k por entropia (E6), gold | orçamento maior ajudaria? |
| E20–E35 | prefixos de 20–35k por entropia, gold | **varredura: onde F1/acc cruzam 0,95×D (piso de orçamento)** |

**Hipótese:** F1(A) ≥ 0,95 × F1(D) com ~18% dos rótulos.

### Novo: robustez multi-semente
A semente de **treino** (`--seed`) varia a aleatoriedade do ajuste fino (init do
cabeçalho de classificação, shuffle intra-época, dropout) e o sorteio do braço C.
O particionamento pool/população e a amostra de avaliação ficam **fixos**
(semente de dados = 42), então os braços continuam pareáveis entre sementes.
Rode **≥3 sementes** (ex.: 42, 7, 123) e reporte média ± desvio — é o item nº 1
do parecer da banca. Os arquivos ficam `e3prime_<braço>_s<semente>.json`
(a semente 42 já está calculada no repositório).

## O que você precisa
1. **GPU**: em *Runtime → Change runtime type*, escolha **T4 GPU** (Colab).
2. **Acesso ao repositório** `GHDaru/activelearning`: um *fine-grained token* do
   GitHub com leitura de conteúdo (cole quando o passo 2 pedir).
3. **`data/dataset.csv`**: já vem versionado no clone — nada a subir.
4. **Persistência (Colab grátis)**: monte o Google Drive e aponte a saída para lá
   (passo 5), para sobreviver a quedas de sessão. Rode **uma semente por sessão**.

Tempo numa T4: **~1,5–2,5 h por semente** (5 braços + varredura, avaliação na
população inteira). Retomada automática pula braço já concluído.

In [ ]:
# 1) GPU disponível?
!nvidia-smi -L || echo 'SEM GPU: mude o runtime para T4 (Colab) ou use cluster GPU (Databricks)'


In [ ]:
# 2) Clonar o repositório (cole seu token quando pedido)
import os
from getpass import getpass
if not os.path.exists('activelearning'):
    tok = getpass('GitHub token (fine-grained, leitura de GHDaru/activelearning): ')
    !git clone https://{tok}@github.com/GHDaru/activelearning.git
%cd activelearning


In [ ]:
# 3) Base de dados — pula se data/dataset.csv já veio no clone
import os
if not os.path.exists('data/dataset.csv'):
    print('Envie o dataset.csv (colunas nm_item, nm_product):')
    from google.colab import files          # em Databricks: use dbfs/Volumes e copie p/ data/
    up = files.upload()
    os.makedirs('data', exist_ok=True)
    fname = next(iter(up))
    os.replace(fname, 'data/dataset.csv')
!wc -l data/dataset.csv


In [ ]:
# 4) Dependências (Colab já traz torch com CUDA; só falta transformers)
%pip -q install transformers scikit-learn
import torch; print('CUDA:', torch.cuda.is_available())


In [ ]:
# 5) E3' multi-semente. Cada semente = uma reamostragem do treino do BERTimbau.
#    Retomada automática: braço concluído (e3prime_<braço>_s<semente>.json) é pulado
#    — se a sessão cair, basta reexecutar esta célula.
#
#    PERSISTÊNCIA (Colab grátis): descomente as 2 linhas do Drive para que os
#    resultados sobrevivam a quedas de sessão. E rode UMA semente por sessão
#    (deixe SEEDS=[42], depois [7], depois [123]).
import os
OUT = 'experiments/e2e3/results'
# from google.colab import drive; drive.mount('/content/drive')
# OUT = '/content/drive/MyDrive/e3prime_results'; os.makedirs(OUT, exist_ok=True)

SEEDS = [42]          # 42 já vem pronta no repo; rode [7] e [123] nas próximas sessões
for S in SEEDS:
    print(f'\n######## SEMENTE DE TREINO {S} ########', flush=True)
    !python experiments/e2e3/run_e3prime.py --arms A,B,C,E,D,E20,E25,E30,E35 \
        --epochs 3 --batch-size 128 --eval-limit 0 --seed {S} --out-dir "{OUT}"

In [ ]:
# 6) Consolidar: cruzamento do critério por semente + média ± desvio entre sementes
import json, glob, os, re
import pandas as pd
OUT = OUT if 'OUT' in dir() else 'experiments/e2e3/results'
rows = []
for p in glob.glob(os.path.join(OUT, 'e3prime_*_s*.json')):
    if p.endswith('_pred.json'):
        continue
    m = re.search(r'e3prime_([A-E][0-9]*)_s(\d+)\.json$', os.path.basename(p))
    if not m:
        continue
    d = json.load(open(p)); d['arm'], d['seed'] = m.group(1), int(m.group(2))
    rows.append(d)
df = pd.DataFrame(rows)[['arm','seed','n_train','accuracy','macro_f1']].sort_values(['seed','n_train'])

for S, g in df.groupby('seed'):
    print(f'\n== semente de treino {S} ==')
    d = g[g.arm == 'D']
    if len(d):
        fd, ad = float(d.macro_f1.iloc[0]), float(d.accuracy.iloc[0])
        print(f'  critério: F1>={0.95*fd:.4f} | acc>={0.95*ad:.4f}')
        for _, r in g.iterrows():
            print(f"  {r.arm:>4} n={r.n_train:>6} F1={r.macro_f1:.4f} "
                  f"[{'OK' if r.macro_f1>=0.95*fd else '--'}] "
                  f"acc={r.accuracy:.4f} [{'OK' if r.accuracy>=0.95*ad else '--'}] "
                  f"({r.n_train/50000:.0%} pool)")

if df.seed.nunique() > 1:
    print('\n== agregado entre sementes (média ± desvio) ==')
    agg = (df.groupby(['arm','n_train'])
             .agg(f1_mean=('macro_f1','mean'), f1_std=('macro_f1','std'),
                  acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
                  k=('seed','nunique')).reset_index().sort_values('n_train'))
    for _, r in agg.iterrows():
        print(f"  {r.arm:>4} n={int(r.n_train):>6} "
              f"F1={r.f1_mean:.4f}±{r.f1_std:.4f} "
              f"acc={r.acc_mean:.4f}±{r.acc_std:.4f} (k={int(r.k)})")
    display(agg)
else:
    print('\n(apenas 1 semente presente — rode outras sementes para média ± desvio)')

os.system(f'zip -qr e3prime_results.zip "{OUT}"')
try:
    from google.colab import files; files.download('e3prime_results.zip')
except ImportError:
    print(f'Databricks/local: resultados em {OUT}')

## Databricks (em vez do Colab)
1. **Cluster**: single node com GPU (ex.: `g4dn.xlarge`/T4 na AWS ou `Standard_NC4as_T4_v3`
   no Azure), runtime *ML com GPU* (torch já incluso).
2. **Código**: *Repos → Add Repo* → `https://github.com/GHDaru/activelearning` (conecte sua
   conta GitHub). Como `data/dataset.csv` está **versionado no repo**, não precisa subir a base.
3. **Dependências**: `%pip install transformers scikit-learn`.
4. **Execução**: as células 5–6 funcionam iguais. Sem os tetos do Colab grátis, você pode
   rodar as 3 sementes de uma vez: `SEEDS = [42, 7, 123]` na célula 5.
5. **Saída**: aponte `OUT` para um Volume (`/Volumes/<catalog>/<schema>/<vol>/e3prime`)
   para persistir entre execuções; o `files.download` do Colab não roda no Databricks.

## Alternativa sem notebook
A imagem Docker do experimento roda os mesmos comandos em qualquer máquina com GPU
(vast.ai/RunPod cobram ~US$ 0,20–0,40/h por uma 3090/4090): ver `docs/e2e3-docker.md`.
Acrescente `--seed 7` / `--seed 123` para as sementes extras.